# Semantic Query Agent

Natural-language-to-SQL agent using the OpenAI Agents SDK through Abacus.AI RouteLLM and the local Microsoft Northwind SQLite database.

#Python Core

-   `from __future__ import annotations`: Allows you to use future type-hinting styles in older Python versions, making type hints evaluate lazily (as strings) rather than immediately when the code runs.

-   `import os`: Talk to operating system to read environment variables or manage file paths.

-   `import re`: Tools to use **regular expressions** to search, match, and change text.

-   `import sqlite3`: Connects to **SQLite database**, because it is lightweight to save and load data locally.

-   `import time`: Work with system time.

-   `from pathlib import Path`: Handle **file and folder paths**.

-   `from typing import Any`: Provides `Any` type hint, which tells Python a variable can be **any data type**


In [4]:
from __future__ import annotations

import os
import re
import sqlite3
import time
from pathlib import Path
from typing import Any

-   `import pandas as pd`: Imports **Pandas**, a tool for working with tables of data, rows, and columns.

-   `from agents import`: Imports specific AI agent classes and tools (like `Agent`, `Runner`, and configuration functions) from OpenAI API(s).

-   `from dotenv import load_dotenv`: Loads secret keys and settings from a hidden `.env` file into your program's environment variables.

-   `from openai import AsyncOpenAI`: Imports the **OpenAI** client designed for **asynchronous code** (running tasks at the same time without blocking).

-   `from pydantic import BaseModel, Field`: Imports **Pydantic**, a tool used to validate data types and structure information using classes and specific field rules.

In [5]:
import pandas as pd
from agents import Agent, Runner, set_default_openai_api, set_default_openai_client, set_tracing_disabled
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

In [7]:
# Set up the environment and load the API key
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'data' / 'northwind.db').exists():
    ROOT_DIR = ROOT_DIR.parent
DATABASE_PATH = ROOT_DIR / 'data' / 'northwind.db'
load_dotenv(ROOT_DIR / '.env')
api_key = os.getenv('ABACUS_API_KEY')
if not api_key:
    raise RuntimeError('Set ABACUS_API_KEY in .env before using the agent.')
    print("Error: OpenAI API key not found. Please set the OPENAI_API_KEY environment variable in your .env file.")
else:
    print("API key loaded successfully.")

API key loaded successfully.


In [ ]:
# RouteLLM reports token counts with the Responses API field names (input_tokens,
# output_tokens) and leaves the Chat Completions fields null. The Agents SDK reads the
# Chat Completions names into its strictly typed Usage model, so those nulls raise
# validation errors. Fill them in from whatever the provider actually returned.
def normalize_usage(completion: Any) -> Any:
    usage = getattr(completion, 'usage', None)
    if usage is None:
        return completion
    prompt_tokens = usage.prompt_tokens
    if prompt_tokens is None:
        prompt_tokens = getattr(usage, 'input_tokens', 0) or 0
    completion_tokens = usage.completion_tokens
    if completion_tokens is None:
        completion_tokens = getattr(usage, 'output_tokens', 0) or 0
    usage.prompt_tokens = prompt_tokens
    usage.completion_tokens = completion_tokens
    if usage.total_tokens is None:
        usage.total_tokens = prompt_tokens + completion_tokens
    return completion

def with_normalized_usage(client: AsyncOpenAI) -> AsyncOpenAI:
    create = client.chat.completions.create

    async def create_with_normalized_usage(*args: Any, **kwargs: Any) -> Any:
        return normalize_usage(await create(*args, **kwargs))

    client.chat.completions.create = create_with_normalized_usage
    return client

client = with_normalized_usage(AsyncOpenAI(base_url=os.getenv('ABACUS_BASE_URL', 'https://routellm.abacus.ai/v1'), api_key=api_key))
set_default_openai_client(client, use_for_tracing=False)
set_default_openai_api('chat_completions')
set_tracing_disabled(True)


Code Summary

-   **Fetches the Database Schema**: The `database_schema()` function connects to a local SQLite database, extracts the names and raw structural code (`CREATE TABLE` SQL statements) for all user-defined tables, and joins them into a single string.
-   **Saves Schema Context**: It executes this function immediately to store the exact database structure in a global variable called `SCHEMA_CONTEXT`.

In [16]:
def database_schema() -> str:
    with sqlite3.connect(DATABASE_PATH) as connection:
        rows = connection.execute("SELECT name, sql FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall()
    return '\n\n'.join(f'{name}: {sql}' for name, sql in rows)

# Saves the schema context of the Northwind database to a variable for use in the agent's instructions.
SCHEMA_CONTEXT = database_schema()

**Defines Structured Output**: The `SQLPlan` class creates a strict template using Pydantic, ensuring the AI model _must_ return exactly three things: the executable SQL code, a business explanation, and the columns used.

In [17]:
class SQLPlan(BaseModel):
    sql: str = Field(description='One read-only SQLite SELECT or WITH query.')
    explanation: str = Field(description='Short business-language explanation of what the query answers.')
    schema_context: str = Field(description='Relevant tables and columns used for the query.')

**Builds the Guardrailed System Prompt**: The `SYSTEM_PROMPT` instructs the AI that its role is a "Semantic Query Agent." It explicitly forbids destructive operations (like `DELETE` or `DROP`), forces the AI to only look at the extracted schema, and gives specific rules for formatting table names with spaces.

**Instantiates the AI Agent**: The final line initializes the `Agent` with its name, a dynamically selected model from environment variables, the system rules, and the strict `SQLPlan` structure to guarantee the format of its answers.

In [21]:
SYSTEM_PROMPT = '''You are Semantic Query Agent. Translate a business question into one safe SQLite query for Microsoft Northwind. Return structured output. Generate exactly one read-only SELECT or WITH query. Never generate INSERT, UPDATE, DELETE, DROP, ALTER, ATTACH, DETACH, PRAGMA, or multiple statements. Quote Northwind identifiers containing spaces, including \"Order Details\". Use only this schema:\n\n''' + SCHEMA_CONTEXT

# Create the agent with the specified model, instructions, and output type.
agent = Agent(name='Semantic Query Agent', 
              model=os.getenv('ABACUS_MODEL', 'route-llm'), 
              instructions=SYSTEM_PROMPT, 
              output_type=SQLPlan
              )

Defines Forbidden Keywords: A regular expression (FORBIDDEN_SQL) blocks destructive SQL commands like INSERT, DELETE, DROP, or database modification statements.

In [ ]:
# Guardrail to prevent the agent from generating unsafe SQL queries.
FORBIDDEN_SQL = re.compile(r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|ATTACH|DETACH|PRAGMA|VACUUM|REINDEX|CREATE|REPLACE)\b', re.IGNORECASE)

Code Summary

-   **Defines Forbidden Keywords**: A regular expression (`FORBIDDEN_SQL`) blocks destructive SQL commands like `INSERT`, `DELETE`, `DROP`, or database modification statements.
-   **Validates SQL Rules**: The `validate_sql()` function trims whitespace and semicolons from the query. It throws a security error if the query does not start with `SELECT` or `WITH`, contains internal semicolons (preventing multi-statement attacks), or contains any forbidden keywords.
-   **Runs the Agent**: The `answer_question()` function submits a business question to the AI agent, waits for the response, and verifies that the output strictly matches the expected `SQLPlan` format.
-   **Executes Safe Read-Only Queries**: The function opens the SQLite database in strict read-only mode (`mode=ro`). It passes the validated SQL query to Pandas (`pd.read_sql_query`) to load the results directly into a structured DataFrame.
-   **Measures and Returns Metrics**: It uses a precise timer (`time.perf_counter`) to calculate query execution speed in milliseconds. Finally, it builds a summary message including the business explanation and total rows returned, bundling everything into a neat data dictionary.

In [24]:
def validate_sql(sql: str) -> str:

    # Strip whitespace and semicolons from the SQL query to ensure it is a single statement. 
    statement = sql.strip().rstrip(';').strip()

    if not statement.upper().startswith(('SELECT', 'WITH')) or ';' in statement or FORBIDDEN_SQL.search(statement):
        raise ValueError('The agent did not return a single read-only SQLite query.')
    return statement

def answer_question(question: str) -> dict[str, object]:

    # Run the agent synchronously to get the SQL plan for the given question.
    plan = Runner.run_sync(agent, question).final_output

    if not isinstance(plan, SQLPlan):
        raise RuntimeError('The agent did not return a SQL plan.')

    # Validate the SQL query to ensure it is safe and read-only.
    sql = validate_sql(plan.sql)

    # Measure the execution time of the SQL query.
    started = time.perf_counter()

    #
    with sqlite3.connect(f'file:{DATABASE_PATH}?mode=ro', uri=True) as connection:
        dataframe = pd.read_sql_query(sql, connection)

    # Measure the execution time in milliseconds.
    execution_ms = (time.perf_counter() - started) * 1000

    # Generate a summary of the number of rows returned by the query.
    row_summary = f'Returned {len(dataframe):,} row' + ('' if len(dataframe) == 1 else 's') + '.'

    return {'response': f'{plan.explanation} {row_summary}', 'sql': sql, 'schema_context': plan.schema_context, 'execution_ms': execution_ms, 'dataframe': dataframe}
